# Visualize Data & Pocket

Load a single protein–ligand complex and show the pocket definition (residues within distance cutoff of the ligand) and pocket center of mass.

In [ ]:
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("SIGMADOCK_DATA_DIR", "../data"))
assert DATA_DIR.exists(), f"Set SIGMADOCK_DATA_DIR or create {DATA_DIR} with benchmark data."

from sigmadock.config import get_experiment_config
from sigmadock.data import SigmaDataset
from sigmadock.datafronts import MetaFront

In [7]:
experiment_name = "posebusters"  # or pdbbind-refined, astex
config = get_experiment_config(experiment_name, DATA_DIR)
datafront = MetaFront([config])
print(f"DataFront: {datafront}, {len(datafront)} complexes")

DataFront: MetaFront(roots=['../data/posebusters_paper/posebusters_benchmark_set'], total_pairs=428), 428 complexes


In [37]:
dataset = SigmaDataset(
    datafront=datafront,
    pocket_distance_cutoff=6.0,
    pocket_com_cutoff=6.0,
    use_esm_embeddings=False,
    pb_check=False,
    alignment_tries=0,
    get_mol_info=True,
    sample_conformer=False,
    skip_bounds_check=True,
    force_retry=False,
)
idx = 123
sample = dataset[idx]
print("Sample keys:", [k for k in sample.keys() if not k.startswith("_")])
print("Pocket CoM:", getattr(sample, "pocket_com", None))

Sample keys: ['pocket_com', 'process_time', 'edge_entity', 'frag_sizes', 'protein_embeddings', 'node_entity', 'frag_atom_idx', 'alignment_energy_delta', 'frag_idx_map', 'edge_attr', 'overconstrained_anchors', 'edge_index', 'frag_counter', 'ref_pos', 'mol_info', 'mask', 'alignment_rmsd', 'dummy_frag_sizes', 'x', 'residue_types', 'overconstrained_dummies', 'triangulation_indexes', 'seed']
Pocket CoM: tensor([[-19.0754, -12.1403, -24.5195]])


## Optional: 3D view

Requires `py3Dmol` and `plotly` (`pip install py3Dmol plotly`).

In [47]:
try:
    from sigmadock.geo.viz import plot_interaction_graph_3d_plotly
    fig = plot_interaction_graph_3d_plotly(
        sample,
        pos_key = "ref_pos",
        show_ligand_virtual = False,
        show_protein_virtual = False,
        show_triangulation = True,
        )
except Exception as e:
    print("Skipping 3D plot:", e)

### Visualize Virtual & Overconstrained Nodes

In [48]:
try:
    from sigmadock.geo.viz import plot_interaction_graph_3d_plotly
    fig = plot_interaction_graph_3d_plotly(
        sample,
        pos_key = "ref_pos",
        show_ligand_virtual = True,
        show_protein_virtual = True,
        show_triangulation = True,
        show_overconstrained = True,
        )
except Exception as e:
    print("Skipping 3D plot:", e)

### Visualize Computational Graph (Subset)

In [49]:
# NOTE: This does not include interaction edges (PL Interaction) which are added at time of graph construction prior to inference.
try:
    from sigmadock.geo.viz import plot_interaction_graph_3d_plotly
    fig = plot_interaction_graph_3d_plotly(
        sample,
        pos_key = "ref_pos",
        show_protein = True,
        show_ligand_virtual = True,
        show_protein_virtual = True,
        show_triangulation = False,
        show_protein_virtual_edges=True,
        show_protein_ligand_virtual_edges=True,
        show_ligand_virtual_edges=True,
        )
except Exception as e:
    print("Skipping 3D plot:", e)